In [1]:
from Environment import *
from DDPG import *
from NN_Module import *
import os

import torch
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from numpy import linalg as LA

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc
from pandapower.plotting.plotly import pf_res_plotly

from loguru import logger

### a simple logger
logger.remove()
logger.add(sys.stderr, level='INFO')

4

In [2]:
env_seed = 7        #10-h  5-h 0-l 1-h 2-l 3-l 4l 7h 8h 9l
episode_num = 200   # the total test episode
step_num = 200      # the longest test step

### create testing environment
injection_bus = np.array([18, 21, 30, 45, 53])-1
pp_net = create_56bus()
env = VoltageCtrl_Env(pp_net, injection_bus)
state, topology, senario = env.reset_topo(seed=env_seed)
topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
# pf_res_plotly(pp_net);

### Some Plot Function

In [3]:
def moving_average(a, n=3):
    # Padding the array to maintain the length after convolution.
    pad = np.pad(a, (n//2, n-1-n//2), mode='edge')
    ret = np.convolve(pad, np.ones(n), mode='valid') / n
    return ret

# plot policy
def policy_plotly(policy_net, topology):
    """
    用 Plotly 绘制各母线的策略曲线，每个子图显示一个母线的 RLC-FT 策略与基线（Linear）策略比较，
    """
    default_colors = pc.qualitative.Plotly  # Plotly 默认颜色序列
    color_linear = default_colors[0]
    color_rlc = default_colors[1]
    fig = make_subplots(rows=1, cols=5,
                        subplot_titles=['Bus 18', 'Bus 21', 'Bus 30', 'Bus 45', 'Bus 53'])
    N = 400
    for i in range(5):
        baseline_vals = []
        policy_vals = []
        for j in range(N):
            # 计算基线控制值：baseline = max(state-1.05, 0) - max(0.95-state, 0)
            state_val = 0.80 + 0.001 * j
            base = np.maximum(state_val - 1.05, 0) - np.maximum(0.95 - state_val, 0)
            baseline_vals.append(-base)  # 取负值
            state_tensor = torch.tensor([[state_val]])
            action_tensor = policy_net[i](state_tensor, topology)
            policy_vals.append(float(-action_tensor.detach().cpu().numpy()[0]))

        baseline_vals = np.array(baseline_vals)
        policy_vals_smoothed = moving_average(np.array(policy_vals), n=20)
        baseline_vals_scaled = 5 * baseline_vals
        
        x_vals = np.linspace(10, 14, N)
        
        # 仅在第一列显示图例，其余子图同组 trace 设为不显示图例
        showlegend = True if i == 0 else False

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=baseline_vals_scaled,
            mode='lines',
            name='Linear',
            legendgroup='Linear',
            showlegend=showlegend,
            line=dict(dash='dash', color=color_linear)
        ), row=1, col=i+1)

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=policy_vals_smoothed,
            mode='lines',
            name='RLC-FT',
            legendgroup='RLC-FT',
            showlegend=showlegend,
            line=dict(color=color_rlc)
        ), row=1, col=i+1)

    # 保证仅在第一个子图显示y轴标题，第三个子图显示x轴标题
    fig.update_yaxes(title_text="Q (MVar)", row=1, col=1)
    fig.update_xaxes(title=dict(text="Voltage (kV)", standoff=25), row=1, col=3)
    fig.update_layout(
        width=1400,
        height=500,
        showlegend=True,
        font=dict(size=16),
        xaxis=dict(
            tickfont=dict(size=12),
            showline=True,
            mirror=True,
            showgrid=True,
        ),
        yaxis=dict(
            tickfont=dict(size=12),
            showline=True,
            mirror=True,
            showgrid=True,
        ),
    )
    
    output_path = os.path.join(Config.data_path, 'images', '56bus', 'policy_plot.pdf')
    import plotly.io as pio
    pio.kaleido.scope.mathjax = None
    fig.write_image(output_path)
    fig.show()


def safe_net_plotly(safe_net):
    """
    用 Plotly 绘制 safe network 策略曲线，每个子图显示一个母线的 Stable-DDPG 与 Linear 比较
    """
    default_colors = pc.qualitative.Plotly  # Plotly 默认颜色序列
    color_linear = default_colors[0]
    color_safe = default_colors[1]
    fig = make_subplots(rows=1, cols=5,
                        subplot_titles=['Bus 18', 'Bus 21', 'Bus 30', 'Bus 45', 'Bus 53'])
    N = 400
    for i in range(len(safe_net)):
        baseline_vals = []
        safe_vals = []
        for j in range(N):
            state_val = 0.80 + 0.001 * j
            base = np.maximum(state_val - 1.05, 0) - np.maximum(0.95 - state_val, 0)
            baseline_vals.append(-base)
            # safe_net[i].get_action 接受列表输入，返回单个数值
            action = safe_net[i].get_action([state_val])
            safe_vals.append(-float(action))
        baseline_vals = np.array(baseline_vals)
        baseline_vals_scaled = 2 * baseline_vals
        x_vals = np.linspace(10, 14, N)
        # 仅在第一列显示图例，其余子图同组 trace 设为不显示图例
        showlegend = True if i == 0 else False

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=baseline_vals_scaled,
            mode='lines',
            name='Linear',
            showlegend=showlegend,
            line=dict(dash='dash', color=color_linear)
        ), row=1, col=i+1)

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=safe_vals,
            mode='lines',
            name='Safe-DDPG',
            showlegend=showlegend,
            line=dict(color=color_safe)
        ), row=1, col=i+1)

    # 保证仅在第一个子图显示y轴标题，第三个子图显示x轴标题
    fig.update_yaxes(title_text="Q (MVar)", row=1, col=1)
    fig.update_xaxes(title=dict(text="Voltage (kV)", standoff=25), row=1, col=3)
    fig.update_layout(
        width=1400,
        height=500,
        showlegend=True,
        xaxis=dict(
            showline=True,
            mirror=True,
            showgrid=True,
        ),
        yaxis=dict(
            showline=True,
            mirror=True,
            showgrid=True,
        ),
    )

    fig.show()


def x_policy_plotly(policy_net):
    """
    用 Plotly 绘制不同拓扑下的 RLC-FT 策略曲线，所有情形绘制在单个图中
    """
    import plotly.graph_objects as go
    fig = go.Figure()
    N = 400
    for i in range(5):
        policy_vals = []
        # 对于每个拓扑情形，通过 reset_topo 获得对应拓扑设定
        state, topo, senario = env.reset_topo(seed=i)
        topo_tensor = torch.cuda.FloatTensor(topo).unsqueeze(0)
        for j in range(N):
            state_tensor = torch.tensor([[0.80 + 0.001 * j]])
            action_tensor = policy_net[2](state_tensor, topo_tensor)
            policy_vals.append(float(-action_tensor.detach().cpu().numpy()[0]))
        policy_vals_smoothed = moving_average(np.array(policy_vals), n=20)
        x_vals = np.linspace(10, 14, N)
        fig.add_trace(go.Scatter(x=x_vals, y=policy_vals_smoothed,
                                 mode='lines',
                                 name=f'Topology {i}'))
    fig.update_layout(
        font=dict(size=16),
        width=700,
        height=500,
        # margin=dict(l=30, r=30, t=30, b=30),   # Uncomment to adjust margins
        margin=dict(r=30,t=30,b=60),
        xaxis_title='Voltage (kV)',
        yaxis_title='Q (MVar)',
        xaxis=dict(
            showgrid=True,
            tickfont=dict(size=12),
        ),
        yaxis=dict(
            tickfont=dict(size=12),
            showgrid=True,
            zeroline=True,
            zerolinecolor='lightgray'
        ),
        legend=dict(
            x=1,
            y=1,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255,255,255,1.0)'
        ),
    )

    fig.show()

### Load model

In [4]:
agent_num = 5
agent_policy_net = []
safe_agent_net = []

### load nn model parameter from saved model 
for i in range(agent_num):
    topology_net = TopologyNet(topology_dim=55, output_dim=1, hidden_dim=256)
    policy_net = FlexiblePolicyNet(env=env, topology_net=topology_net, obs_dim=1, action_dim=1, hidden_dim=2048).to(device)
    agent_policy_net.append(policy_net)

for i in range(agent_num):
    policy_net = SafePolicyNetwork(env=env, obs_dim=1, action_dim=1, hidden_dim=100).to(device)
    safe_agent_net.append(policy_net)

for i in range(agent_num):
    #value_net_dict = torch.load(f'check_points/value_net/2023-06-19/Step_200_Seed_12_a{i}.pth')
    #policy_net_dict = torch.load(f'check_points/policy_net/2023-07-05/Step_300_Seed_45_a{i}.pth')
    # policy_net_dict = torch.load(f'check_points/policy_net/2023-08-15/Step_900_Seed_33_a{i}.pth')
    #policy_net_dict = torch.load(f'check_points/policy_net/2023-09-21/Step_900_Seed_10_a{i}.pth')
    # policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_600_Seed_12_a{i}.pth'))
    policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_500_Seed_4_a{i}.pth'))

    agent_policy_net[i].load_state_dict(policy_net_dict)

for i in range(agent_num):
    #value_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/2023-06-19/SafeDDPG_value_Step_200_a{i}.pth')
    policy_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/stable_ddpg/policy_net_checkpoint_a{i}.pth')
    safe_agent_net[i].load_state_dict(policy_net_dict)

In [6]:
policy_plotly(agent_policy_net, topology)
# safe_net_plotly(safe_agent_net)

### Flexible NN Contoller

In [7]:
### test our controller
voltage = []
q = []
cost = []
success_list = []
fail_list = []
entire_list = []
control_cost = []
reward_list = []
object_cost = []
voltage_trajs = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    cost = []
    abnormal_stop = False

    voltage_episode = []   # stores full voltage vectors for this episode

    for step in range(step_num):
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
            action_agent = 0.6 * action_agent.detach().cpu().numpy()[0] #0.7
            action.append(action_agent)

        action = last_action - np.asarray(action)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            success_list.append((episode,step))
            logger.success('episode {} stable at {} steps',success_list[-1][0], success_list[-1][1])
            break

        voltage.append(state)
        voltage_episode.append(state.copy())  # record full state vector

        q.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    reward_list.append(episode_reward)
    control_cost.append(episode_control)
    object_cost.append(np.sum(cost))
    if len(voltage_episode) > 0 and senario == 0:
        voltage_trajs.append(np.vstack(voltage_episode))

    if (not done) and (abnormal_stop == False):
        entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(success_list))
logger.info('total fail episode is {}', len(fail_list))
logger.info('number of finished at entire episode is {}', len(entire_list))

2025-06-28 17:30:54.253 | SUCCESS  | __main__:<module>:51 - episode 0 stable at 8 steps
2025-06-28 17:30:54.459 | SUCCESS  | __main__:<module>:51 - episode 1 stable at 2 steps
2025-06-28 17:30:54.967 | SUCCESS  | __main__:<module>:51 - episode 2 stable at 9 steps
2025-06-28 17:30:55.263 | SUCCESS  | __main__:<module>:51 - episode 3 stable at 6 steps
2025-06-28 17:30:55.556 | SUCCESS  | __main__:<module>:51 - episode 4 stable at 6 steps
2025-06-28 17:30:56.182 | SUCCESS  | __main__:<module>:51 - episode 5 stable at 15 steps
2025-06-28 17:30:56.283 | SUCCESS  | __main__:<module>:51 - episode 6 stable at 1 steps
2025-06-28 17:30:57.856 | SUCCESS  | __main__:<module>:51 - episode 7 stable at 39 steps
2025-06-28 17:30:58.718 | SUCCESS  | __main__:<module>:51 - episode 8 stable at 20 steps
2025-06-28 17:30:59.269 | SUCCESS  | __main__:<module>:51 - episode 9 stable at 13 steps
2025-06-28 17:30:59.868 | SUCCESS  | __main__:<module>:51 - episode 10 stable at 14 steps
2025-06-28 17:31:00.217 | 

In [8]:
success_list = np.array(success_list)
print('average recovery step is:')
print(np.mean(success_list[:,1]))
print(np.std(success_list[:,1]))
print('average reactive power cost is:')
print(np.mean(control_cost))
print(np.std(control_cost))
print('the total cost is:')
# print(object_cost)
print(np.mean(object_cost))
print(np.std(object_cost))

import plotly.express as px

# ---------- 3. plotting ----------
bus_idx = 2                                   # choose bus (0-based)
max_len  = max(a.shape[0] for a in voltage_trajs)

# build (episodes × max_len) matrix with NaN padding
traj_mat = np.full((len(voltage_trajs), max_len), np.nan)
for i, ep in enumerate(voltage_trajs):
    traj_mat[i, :ep.shape[0]] = ep[:, bus_idx]

mean_traj = np.nanmean(traj_mat, axis=0)
min_traj  = np.nanmin(traj_mat, axis=0)
max_traj  = np.nanmax(traj_mat, axis=0)

# pick one color from Plotly's qualitative palette
base_color = px.colors.qualitative.Plotly[0]  # '#1f77b4' (blue)

def hex_to_rgba(hex_color, alpha):
    """#1f77b4 -> rgba(31,119,180,alpha)"""
    h = hex_color.lstrip('#')
    r, g, b = tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
    return f'rgba({r},{g},{b},{alpha})'

fill_color = hex_to_rgba(base_color, 0.25)    # translucent blue

fig = go.Figure()

# 1) lower bound (invisible line – starting edge of the band)
fig.add_trace(
    go.Scatter(
        x=np.arange(max_len),
        y=min_traj,
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        hoverinfo="skip",
        name="min",
    )
)

# 2) upper bound with 'tonexty' fill – creates the shaded band
fig.add_trace(
    go.Scatter(
        x=np.arange(max_len),
        y=max_traj,
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor=fill_color,
        showlegend=False,
        hoverinfo="skip",
        name="max",
    )
)

# 3) mean curve on top, same hue but solid & thicker
fig.add_trace(
    go.Scatter(
        x=np.arange(max_len),
        y=mean_traj,
        mode="lines",
        name=f"Mean Voltage (Bus {bus_idx + 1})",
        line=dict(color=base_color, width=3),
    )
)

fig.update_layout(
    title=f"Voltage Trajectories on Bus {bus_idx + 1}",
    xaxis_title="Step",
    yaxis_title="Voltage (p.u.)",
    template="plotly_white",
)

fig.show()

average recovery step is:
9.235
9.661251212964084
average reactive power cost is:
37.408523721354776
71.32292213302807
the total cost is:
155.75102431880367
178.74615110890434


In [ ]:
### test our controller without topology change
voltage_ = []
q_ = []
cost_ = []
success_list_ = []
fail_list_ = []
entire_list_ = []
control_cost_ = []
reward_list_ = []
object_cost_list_ = []
voltage_trajs_ = []

def apply_hybrid_error(true_topology: torch.Tensor, error_percentage: float) -> torch.Tensor:
    """
    Applies a hybrid error model to a topology tensor.

    For a specified percentage of connected lines, this function will either
    set the line's susceptance to zero (simulating disconnection) or perturb
    it with multiplicative Gaussian noise (simulating parameter corruption).

    Args:
        true_topology: The original, correct topology tensor.
        error_percentage: The fraction of connected lines to apply an error to (e.g., 0.3 for 30%).

    Returns:
        A new topology tensor with the errors applied.
    """
    # Create a copy to modify
    believed_topology = true_topology.clone()

    # 1. Find the indices of all connected lines (non-zero elements)
    connected_indices = torch.nonzero(true_topology)
    num_connected_lines = connected_indices.size(0)

    # 2. Determine how many lines to apply an error to
    num_lines_to_corrupt = int(num_connected_lines * error_percentage)

    # 3. Randomly select the indices of the lines that will have an error
    corruption_indices_perm = torch.randperm(num_connected_lines)
    indices_to_corrupt = connected_indices[corruption_indices_perm[:num_lines_to_corrupt]]

    # 4. Apply the hybrid error to each selected line
    for idx_tuple in indices_to_corrupt:
        # This index tuple is something like (batch, channel, row, col)
        # We need to convert it to a simple tuple for indexing
        idx = tuple(idx_tuple.tolist())

        # 50% chance to simulate disconnection (set to zero)
        if torch.rand(1).item() < 0.8:
            believed_topology[idx] = 0.0
        # 50% chance to simulate parameter corruption
        else:
            original_value = believed_topology[idx]
            # Get a random perturbation factor from N(mean=1.0, std=0.5)
            perturb_factor = torch.normal(mean=1.0, std=1.0, size=(1,)).item()
            # Apply perturbation, ensuring the value is not negative
            believed_topology[idx] = max(0.0, original_value * perturb_factor)

    return believed_topology

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    believed_topology = apply_hybrid_error(topology, error_percentage=0.5)

    # print(f'Topology for episode {episode}: {believed_topology}')

    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    cost_ = []
    abnormal_stop = False
    voltage_episode = []   # stores full voltage vectors for this episode

    for step in range(step_num):
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), believed_topology)
            action_agent = 0.6* action_agent.detach().cpu().numpy()[0]
            action.append(action_agent)

        action = last_action - np.asarray(action)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            fail_list_.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            fail_list_.append((episode,step))
            abnormal_stop = True
            break
        if done:
            success_list_.append((episode,step))
            logger.success('episode {} stable at {} steps',success_list_[-1][0], success_list_[-1][1])
            break

        voltage_.append(state)
        voltage_episode.append(state.copy())      # record full state vector

        q_.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost_.append(-reward)
        
        episode_control += LA.norm(action, 2)

    reward_list_.append(episode_reward)
    control_cost_.append(episode_control)
    object_cost_list_.append(np.sum(cost_))
    if len(voltage_episode) > 0 and senario == 0:
        voltage_trajs_.append(np.vstack(voltage_episode))

    if (not done) and (abnormal_stop == False):
        entire_list_.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(success_list_))
logger.info('total fail episode is {}', len(fail_list_))
logger.info('number of finished at entire episode is {}', len(entire_list_))

Topology for episode 0: tensor([[ 0.0000e+00,  0.0000e+00,  2.6219e+00,  2.1735e+00,  0.0000e+00,
          0.0000e+00,  2.9142e+00,  0.0000e+00,  0.0000e+00,  3.6519e+00,
          0.0000e+00,  0.0000e+00,  3.5605e+00,  0.0000e+00,  0.0000e+00,
          6.8287e+00,  1.2674e+00,  1.2691e+00,  2.7397e+00,  0.0000e+00,
          1.3166e+00,  1.6883e+00,  0.0000e+00,  1.3457e+00,  2.2102e+00,
          2.3706e+00,  4.6686e+00,  0.0000e+00,  0.0000e+00,  3.5194e+00,
          1.8486e+00,  0.0000e+00,  5.3511e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00, -7.1930e-03,  3.8447e+00,  9.4921e-01,  3.2915e+00,
          2.3829e+00,  0.0000e+00, -5.2784e-04,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  4.6686e+00,  4.0955e-04,  3.3890e+00,
          0.0000e+00,  0.0000e+00,  1.1731e+00,  0.0000e+00,  3.5776e-03]],
       device='cuda:0')


2025-06-24 17:33:56.470 | SUCCESS  | __main__:<module>:102 - episode 0 stable at 8 steps


Topology for episode 1: tensor([[ 0.0000e+00,  2.1204e+00,  1.9138e+00,  1.5865e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  8.2792e-03,  0.0000e+00,  2.0426e+00,
          0.0000e+00,  0.0000e+00,  7.6283e+00,  1.6656e+00,  0.0000e+00,
          0.0000e+00,  9.2510e-01,  9.2638e-01,  1.9998e+00,  6.9575e+00,
          9.6104e-01,  1.2323e+00, -1.3860e-03,  9.8224e-01,  1.6133e+00,
          1.7304e+00,  3.4078e+00,  8.3506e-01,  2.8790e+00,  0.0000e+00,
          1.3493e+00,  9.1496e+00,  3.9060e+00,  2.4466e+00,  0.0000e+00,
          6.3904e-03,  0.0000e+00,  2.8064e+00,  0.0000e+00,  0.0000e+00,
          1.7394e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  8.6856e-01,
          0.0000e+00,  9.9690e-01,  3.4078e+00,  0.0000e+00,  2.4738e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00]],
       device='cuda:0')


2025-06-24 17:33:56.727 | SUCCESS  | __main__:<module>:102 - episode 1 stable at 3 steps


Topology for episode 2: tensor([[ 3.7621e+00,  4.6339e+00,  4.1825e+00,  3.4672e+00,  0.0000e+00,
          5.3642e+00,  1.3114e+01, -4.8300e-03,  3.3099e+00,  4.4639e+00,
          9.2415e-04,  0.0000e+00,  0.0000e+00,  0.0000e+00, -6.4603e-03,
          1.0893e+01,  0.0000e+00,  0.0000e+00,  4.3703e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  9.3926e-03,  0.0000e+00,  0.0000e+00,
          3.7816e+00,  7.4473e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          2.9488e+00,  1.9996e+01,  0.0000e+00,  5.3468e+00,  6.9841e+00,
          2.5926e+00,  0.0000e+00,  0.0000e+00, -7.6761e-03,  5.2506e+00,
          3.8012e+00,  4.5656e+00,  8.9551e+00,  7.4855e+00,  1.8982e+00,
          0.0000e+00,  0.0000e+00,  4.5931e-01, -6.3655e-03,  0.0000e+00,
         -1.1420e-04,  0.0000e+00,  0.0000e+00,  4.3703e+00,  1.3153e-03]],
       device='cuda:0')


...

2025-06-24 17:35:47.702 | SUCCESS  | __main__:<module>:102 - episode 197 stable at 11 steps


Topology for episode 198: tensor([[0.0000e+00, 0.0000e+00, 2.4833e+00, 2.0586e+00, 0.0000e+00, 1.8519e+00,
         2.7601e+00, 0.0000e+00, 0.0000e+00, 2.6504e+00, 8.6755e-01, 2.2989e+00,
         3.3723e+00, 0.0000e+00, 7.5018e-03, 0.0000e+00, 1.2004e+00, 1.2021e+00,
         0.0000e+00, 1.6753e+01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 2.3487e+00, 3.7357e+00, 3.3334e+00,
         0.0000e+00, 7.3269e-03, 5.0683e+00, 0.0000e+00, 0.0000e+00, 2.2583e-04,
         4.5992e-01, 0.0000e+00, 0.0000e+00, 3.1176e+00, 2.2570e+00, 7.3368e-01,
         5.3171e+00, 4.4445e+00, 1.1270e+00, 2.2868e+00, 0.0000e+00, 4.4219e+00,
         0.0000e+00, 3.2099e+00, 0.0000e+00, 5.0982e+00, 1.1111e+00, 2.5949e+00,
         0.0000e+00]], device='cuda:0')


2025-06-24 17:35:47.980 | SUCCESS  | __main__:<module>:102 - episode 198 stable at 5 steps


Topology for episode 199: tensor([[ 2.4153e+00,  2.9751e+00,  0.0000e+00,  2.2260e+00,  2.0111e+00,
          2.0025e+00,  2.9846e+00,  1.1744e+00,  2.1251e+00,  2.8659e+00,
          0.0000e+00,  2.4858e+00,  3.6465e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  1.2980e+00,  0.0000e+00,  2.8058e+00,  6.3217e+00,
          1.3484e+00,  1.7291e+00,  0.0000e+00,  1.3782e+00,  0.0000e+00,
          0.0000e+00,  4.7814e+00,  2.5397e+00,  0.0000e+00,  0.0000e+00,
          1.8932e+00,  0.0000e+00,  5.4804e+00,  3.4328e+00,  4.4840e+00,
         -4.1152e-03,  0.0000e+00,  0.0000e+00,  9.7215e-01,  0.0000e+00,
          2.4405e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  4.7814e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  5.5126e+00,  1.2015e+00,  0.0000e+00,  0.0000e+00]],
       device='cuda:0')


2025-06-24 17:35:48.509 | SUCCESS  | __main__:<module>:102 - episode 199 stable at 10 steps
2025-06-24 17:35:48.510 | INFO     | __main__:<module>:128 - total success epsisode is 200
2025-06-24 17:35:48.510 | INFO     | __main__:<module>:129 - total fail episode is 0
2025-06-24 17:35:48.511 | INFO     | __main__:<module>:130 - number of finished at entire episode is 0


In [49]:
success_list_ = np.array(success_list_)
print('average recovery step is:')
print(np.mean(success_list_[:,1]))
print(np.std(success_list_[:,1]))
print('average reactive power cost is:')
print(np.mean(control_cost_))
print(np.std(control_cost_))
print('the total cost is:')
print(np.mean(object_cost_list_))
print(np.std(object_cost_list_))

import os
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# ------------------------------------------------------------
# Nature-style figure configuration
# ------------------------------------------------------------
NATURE_CONFIG = {
    "width": 1800,
    "height": 900,
    "font_base": 28,
    "font_title": 32,
    "font_axis": 24,
    "font_legend": 24,
    "dpi": 300,          # only used indirectly through scale
}

# ------------------------------------------------------------
# helper functions
# ------------------------------------------------------------
def build_stats(trajs, bus_idx, max_len):
    mat = np.full((len(trajs), max_len), np.nan)
    for i, ep in enumerate(trajs):
        mat[i, :ep.shape[0]] = ep[:, bus_idx]
    return {
        "mean": np.nanmean(mat, axis=0),
        "min":  np.nanmin(mat, axis=0),
        "max":  np.nanmax(mat, axis=0)
    }

def hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip('#')
    r, g, b = (int(h[i:i+2], 16) for i in (0, 2, 4))
    return f'rgba({r},{g},{b},{alpha})'

# ------------------------------------------------------------
# choose bus & palette
# ------------------------------------------------------------
bus_idx = 2               # Bus-30
color_perfect = "#1b9e77" # teal   – perfect topology
color_error   = "#d95f02" # orange – 50 % error

max_len = max(
    max(a.shape[0] for a in voltage_trajs),
    max(a.shape[0] for a in voltage_trajs_)
)

stats_perfect = build_stats(voltage_trajs,  bus_idx, max_len)
stats_error   = build_stats(voltage_trajs_, bus_idx, max_len)

# ------------------------------------------------------------
# build figure
# ------------------------------------------------------------
fig = go.Figure()

# ---- shaded band : perfect topology ----
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_perfect["min"],
    mode="lines", line=dict(width=0),
    showlegend=False, hoverinfo="skip",
    legendgroup="perfect"
))
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_perfect["max"],
    mode="lines", line=dict(width=0),
    fill="tonexty", fillcolor=hex_to_rgba(color_perfect, 0.25),
    showlegend=False, hoverinfo="skip",
    legendgroup="perfect"
))

# ---- shaded band : 50 % error ----
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_error["min"],
    mode="lines", line=dict(width=0),
    showlegend=False, hoverinfo="skip",
    legendgroup="error"
))
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_error["max"],
    mode="lines", line=dict(width=0),
    fill="tonexty", fillcolor=hex_to_rgba(color_error, 0.25),
    showlegend=False, hoverinfo="skip",
    legendgroup="error"
))

# ---- mean curves ----
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_perfect["mean"],
    mode="lines",
    name="Perfect information (mean)",
    line=dict(color=color_perfect, width=5),
    legendgroup="perfect"
))
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_error["mean"],
    mode="lines",
    name="50 % error information (mean)",
    line=dict(color=color_error, width=5),
    legendgroup="error"
))

# ---- safety threshold ----
fig.add_trace(go.Scatter(
    x=[0, max_len - 1], y=[0.95, 0.95],
    mode="lines", name="Safety limit 0.95 p.u.",
    line=dict(color="black", dash="dash"),
    hoverinfo="skip"
))

# ------------------------------------------------------------
# layout: Nature style
# ------------------------------------------------------------
fig.update_layout(
    width=NATURE_CONFIG["width"],
    height=NATURE_CONFIG["height"],
    template="plotly_white",
    title=dict(
        text=f"Average Voltage on Bus 30: perfect vs 50 % error topology information",
        font=dict(size=NATURE_CONFIG["font_title"])
    ),
    font=dict(size=NATURE_CONFIG["font_base"]),
    legend=dict(
        x=0.99, y=0.99, xanchor="right", yanchor="top",
        font=dict(size=NATURE_CONFIG["font_legend"])
    ),
    xaxis=dict(
        title="Step",
        titlefont=dict(size=NATURE_CONFIG["font_axis"]),
        tickfont=dict(size=NATURE_CONFIG["font_axis"]),
        range=[0, 18],
    ),
    yaxis=dict(
        title="Voltage (p.u.)",
        titlefont=dict(size=NATURE_CONFIG["font_axis"]),
        tickfont=dict(size=NATURE_CONFIG["font_axis"])
    )
)

fig.show()

# ------------------------------------------------------------
# export high-resolution images
# ------------------------------------------------------------
pio.kaleido.scope.mathjax = None  # speed-up export

output_dir = os.path.join(Config.data_path, "images", "56bus")
os.makedirs(output_dir, exist_ok=True)

pdf_path = os.path.join(output_dir, "voltage_plot.pdf")
png_path = os.path.join(output_dir, "voltage_plot.png")

fig.write_image(pdf_path,  width=NATURE_CONFIG["width"],
                           height=NATURE_CONFIG["height"],
                           scale=2)   # 2 × 1800 px ≈ 300 dpi
fig.write_image(png_path,  width=NATURE_CONFIG["width"],
                           height=NATURE_CONFIG["height"],
                           scale=2)

average recovery step is:
11.31
11.695037409089378
average reactive power cost is:
45.61502732903127
89.11256940616249
the total cost is:
189.06468415553644
220.01792684506054


C:\Users\wdyao\AppData\Local\Temp\ipykernel_21272\1156879876.py:38: RuntimeWarning:

Mean of empty slice

C:\Users\wdyao\AppData\Local\Temp\ipykernel_21272\1156879876.py:39: RuntimeWarning:

All-NaN slice encountered

C:\Users\wdyao\AppData\Local\Temp\ipykernel_21272\1156879876.py:40: RuntimeWarning:

All-NaN slice encountered



In [22]:
def apply_hybrid_error(true_topology: torch.Tensor, error_percentage: float) -> torch.Tensor:
    # Create a copy to modify
    believed_topology = true_topology.clone()

    # 1. Find the indices of all connected lines (non-zero elements)
    connected_indices = torch.nonzero(true_topology)
    num_connected_lines = connected_indices.size(0)

    # 2. Determine how many lines to apply an error to
    num_lines_to_corrupt = int(num_connected_lines * error_percentage)

    # 3. Randomly select the indices of the lines that will have an error
    corruption_indices_perm = torch.randperm(num_connected_lines)
    indices_to_corrupt = connected_indices[corruption_indices_perm[:num_lines_to_corrupt]]

    # 4. Apply the hybrid error to each selected line
    for idx_tuple in indices_to_corrupt:
        # This index tuple is something like (batch, channel, row, col)
        # We need to convert it to a simple tuple for indexing
        idx = tuple(idx_tuple.tolist())

        # 50% chance to simulate disconnection (set to zero)
        if torch.rand(1).item() < 0.5:
            believed_topology[idx] = 0.0
        # 50% chance to simulate parameter corruption
        else:
            original_value = believed_topology[idx]
            # Get a random perturbation factor from N(mean=1.0, std=0.5)
            perturb_factor = torch.normal(mean=1.0, std=0.5, size=(1,)).item()
            # Apply perturbation, ensuring the value is not negative
            believed_topology[idx] = max(0.0, original_value * perturb_factor)

    return believed_topology

# ============================================================================ #
#                         H Y P E R   P A R A M E T E R S                      #
# ---------------------------------------------------------------------------- #
loss_prob_arr     = 0.5          # scalar or array → packet-loss probability
min_delay_arr     = 0            # make delay >= 3 so most episodes feel it
max_delay_arr     = 3
noise_prob_arr    = 0.5          # chance a delivered packet is corrupted
# ============================================================================ #

class CommChannel:
    """Packet-loss / delay / corruption plus an initial black-out window."""
    def __init__(self, loss_prob, min_delay, max_delay, noise_prob):
        self.loss_prob   = loss_prob
        self.min_delay   = min_delay
        self.max_delay   = max_delay
        self.noise_prob  = noise_prob
        self.queue       = []               # [(arrival_step, topo_tensor), …]
        self.current     = None             # last successfully received topology

    def step(self, true_topology: torch.Tensor, t: int) -> torch.Tensor:
        # 1) deliver everything whose arrival time is now
        while self.queue and self.queue[0][0] == t:
            _, topo = self.queue.pop(0)
            self.current = topo.clone()

        # 2) send a NEW packet 
        if np.random.rand() > self.loss_prob:
            delay  = np.random.randint(self.min_delay, self.max_delay + 1)
            packet = true_topology.clone()
            if np.random.rand() < self.noise_prob:
                packet = apply_hybrid_error(packet, error_percentage=0.5)
            self.queue.append((t + delay, packet))
            self.queue.sort(key=lambda x: x[0])

        # 3) return whatever we currently know (may be None)
        return self.current
# --------------------------------------------------------------------------- #


# ============================== result buffers ============================== #
voltage_loss          = []
q_loss                = []
reward_list_loss      = []
control_cost_loss     = []
object_cost_list_loss = []
success_list_loss     = []
fail_list_loss        = []
entire_list_loss      = []
voltage_trajs_loss    = []
# =========================================================================== #

for episode in range(episode_num):
    state, true_topology, scenario = env.reset_topo(seed=episode)
    true_topology = torch.cuda.FloatTensor(true_topology).unsqueeze(0)

    # ------------------- build one channel per agent -----------------------
    channels = []
    for i in range(agent_num):
        ch = CommChannel(
            loss_prob   = loss_prob_arr  if np.ndim(loss_prob_arr)  == 0 else loss_prob_arr[i],
            min_delay   = min_delay_arr  if np.ndim(min_delay_arr)  == 0 else min_delay_arr[i],
            max_delay   = max_delay_arr  if np.ndim(max_delay_arr)  == 0 else max_delay_arr[i],
            noise_prob  = noise_prob_arr if np.ndim(noise_prob_arr) == 0 else noise_prob_arr[i],
        )

        # ----------- bootstrap topology knowledge at t = 0 ----------------
        if np.random.rand() > loss_prob_arr:
            if np.random.rand() < noise_prob_arr:
                # apply hybrid error to the true topology
                ch.current = apply_hybrid_error(true_topology.clone(), 0.5)
            else:
                # no error, just copy the true topology
                ch.current = true_topology.clone()
        # "none": leave ch.current = None
        channels.append(ch)
    # ----------------------------------------------------------------------

    last_action      = np.zeros((agent_num, 1))
    episode_reward   = 0.0
    episode_control  = 0.0
    step_cost_list   = []
    abnormal_stop    = False
    voltage_episode  = []

    for step in range(step_num):

        # -------- per-agent, possibly None, believed topologies -----------
        believed_topos = [ch.step(true_topology, step) for ch in channels]

        # When an agent still has no information (None), you can choose:
        #   • Skip its control (keep previous action),
        #   • Feed a zero tensor,
        #   • Feed some default known topology.
        # Here we fall back to a zero tensor with the correct shape.
        for k, topo in enumerate(believed_topos):
            if topo is None:
                believed_topos[k] = torch.zeros_like(true_topology)

        # -------------------- compute agent actions -----------------------
        actions = []
        for i in range(agent_num):
            a_i = agent_policy_net[i](
                torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0),
                believed_topos[i]
            )
            a_i = 0.6 * a_i.detach().cpu().numpy()[0]
            actions.append(a_i)
        actions = last_action - np.asarray(actions)
        last_action = np.copy(actions)
        # -----------------------------------------------------------------

        # ------------------------ environment step ------------------------
        try:
            next_state, reward, done = env.step(actions)
        except pp.powerflow.LoadflowNotConverged:
            logger.error('power flow not converge at episode {} step {}', episode, step)
            fail_list_loss.append((episode, step))
            abnormal_stop = True
            break

        if np.min(next_state) < 0.75 or np.max(next_state) > 1.25:
            logger.warning('episode {} step {} voltage violation', episode, step)
            fail_list_loss.append((episode, step))
            abnormal_stop = True
            break

        if done:
            success_list_loss.append((episode, step))
            logger.success('episode {} stable at {} steps', episode, step)
            break
        # -----------------------------------------------------------------

        # --------------------------- bookkeeping --------------------------
        voltage_loss.append(state)
        voltage_episode.append(state.copy())
        q_loss.append(actions)

        state = next_state
        episode_reward  += reward
        step_cost_list.append(-reward)
        episode_control += np.linalg.norm(actions, 2)
        # -----------------------------------------------------------------

    # ================= episode-level bookkeeping ==========================
    reward_list_loss.append(episode_reward)
    control_cost_loss.append(episode_control)
    object_cost_list_loss.append(np.sum(step_cost_list))
    if voltage_episode and scenario == 0:
        voltage_trajs_loss.append(np.vstack(voltage_episode))

    if (not done) and (not abnormal_stop):
        entire_list_loss.append(episode)
        logger.info('Episode {} finished with full horizon', episode)

# =========================== final statistics ============================== #
logger.info('total success episodes (lossy): {}', len(success_list_loss))
logger.info('total fail episodes    (lossy): {}', len(fail_list_loss))
logger.info('episodes ran full horizon (lossy): {}', len(entire_list_loss))

2025-06-28 18:38:58.676 | SUCCESS  | __main__:<module>:163 - episode 0 stable at 8 steps
2025-06-28 18:38:58.916 | SUCCESS  | __main__:<module>:163 - episode 1 stable at 2 steps
2025-06-28 18:38:59.557 | SUCCESS  | __main__:<module>:163 - episode 2 stable at 10 steps
2025-06-28 18:38:59.897 | SUCCESS  | __main__:<module>:163 - episode 3 stable at 6 steps
2025-06-28 18:39:00.320 | SUCCESS  | __main__:<module>:163 - episode 4 stable at 8 steps
2025-06-28 18:39:01.251 | SUCCESS  | __main__:<module>:163 - episode 5 stable at 17 steps
2025-06-28 18:39:01.579 | SUCCESS  | __main__:<module>:163 - episode 6 stable at 4 steps
2025-06-28 18:39:03.946 | SUCCESS  | __main__:<module>:163 - episode 7 stable at 46 steps
2025-06-28 18:39:04.900 | SUCCESS  | __main__:<module>:163 - episode 8 stable at 20 steps
2025-06-28 18:39:06.778 | SUCCESS  | __main__:<module>:163 - episode 9 stable at 38 steps
2025-06-28 18:39:07.582 | SUCCESS  | __main__:<module>:163 - episode 10 stable at 16 steps
2025-06-28 18:

In [24]:
success_list_loss = np.array(success_list_loss)
print('average recovery step is:')
print(np.mean(success_list_loss[:,1]))
print(np.std(success_list_loss[:,1]))
print('average reactive power cost is:')
print(np.mean(control_cost_loss))
print(np.std(control_cost_loss))
print('the total cost is:')
print(np.mean(object_cost_list_loss))
print(np.std(object_cost_list_loss))

import os
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# ------------------------------------------------------------
# Nature-style figure configuration
# ------------------------------------------------------------
NATURE_CONFIG = {
    "width": 1800,
    "height": 900,
    "font_base": 28,
    "font_title": 32,
    "font_axis": 24,
    "font_legend": 24,
    "dpi": 300,          # only used indirectly through scale
}

# ------------------------------------------------------------
# helper functions
# ------------------------------------------------------------
def build_stats(trajs, bus_idx, max_len):
    mat = np.full((len(trajs), max_len), np.nan)
    for i, ep in enumerate(trajs):
        mat[i, :ep.shape[0]] = ep[:, bus_idx]
    return {
        "mean": np.nanmean(mat, axis=0),
        "min":  np.nanmin(mat, axis=0),
        "max":  np.nanmax(mat, axis=0)
    }

def hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip('#')
    r, g, b = (int(h[i:i+2], 16) for i in (0, 2, 4))
    return f'rgba({r},{g},{b},{alpha})'

# ------------------------------------------------------------
# choose bus & palette
# ------------------------------------------------------------
bus_idx = 2               # Bus-30
color_perfect = "#1b9e77" # teal   – 0% loss
color_error   = "#d95f02" # orange – 50 % loss

max_len = max(
    max(a.shape[0] for a in voltage_trajs),
    max(a.shape[0] for a in voltage_trajs_loss)
)

stats_perfect = build_stats(voltage_trajs,  bus_idx, max_len)
stats_error   = build_stats(voltage_trajs_loss, bus_idx, max_len)

# ------------------------------------------------------------
# build figure
# ------------------------------------------------------------
fig = go.Figure()

# ---- shaded band : perfect topology ----
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_perfect["min"],
    mode="lines", line=dict(width=0),
    showlegend=False, hoverinfo="skip",
    legendgroup="perfect"
))
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_perfect["max"],
    mode="lines", line=dict(width=0),
    fill="tonexty", fillcolor=hex_to_rgba(color_perfect, 0.25),
    showlegend=False, hoverinfo="skip",
    legendgroup="perfect"
))

# ---- shaded band : 50 % error ----
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_error["min"],
    mode="lines", line=dict(width=0),
    showlegend=False, hoverinfo="skip",
    legendgroup="error"
))
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_error["max"],
    mode="lines", line=dict(width=0),
    fill="tonexty", fillcolor=hex_to_rgba(color_error, 0.25),
    showlegend=False, hoverinfo="skip",
    legendgroup="error"
))

# ---- mean curves ----
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_perfect["mean"],
    mode="lines",
    name="Perfect information (mean)",
    line=dict(color=color_perfect, width=5),
    legendgroup="perfect"
))
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_error["mean"],
    mode="lines",
    name="50 % error information (mean)",
    line=dict(color=color_error, width=5),
    legendgroup="error"
))

# ---- safety threshold ----
fig.add_trace(go.Scatter(
    x=[0, max_len - 1], y=[0.95, 0.95],
    mode="lines", name="Safety limit 0.95 p.u.",
    line=dict(color="black", dash="dash"),
    hoverinfo="skip"
))

# ------------------------------------------------------------
# layout: Nature style
# ------------------------------------------------------------
fig.update_layout(
    width=NATURE_CONFIG["width"],
    height=NATURE_CONFIG["height"],
    template="plotly_white",
    title=dict(
        text=f"Average Voltage on Bus 30: 0% vs 50 % topology information loss",
        font=dict(size=NATURE_CONFIG["font_title"])
    ),
    font=dict(size=NATURE_CONFIG["font_base"]),
    legend=dict(
        x=1, y=0.99, xanchor="right", yanchor="top",
        font=dict(size=NATURE_CONFIG["font_legend"])
    ),
    xaxis=dict(
        title="Step",
        titlefont=dict(size=NATURE_CONFIG["font_axis"]),
        tickfont=dict(size=NATURE_CONFIG["font_axis"]),
        range=[0, 18],
    ),
    yaxis=dict(
        title="Voltage (p.u.)",
        titlefont=dict(size=NATURE_CONFIG["font_axis"]),
        tickfont=dict(size=NATURE_CONFIG["font_axis"])
    )
)

fig.show()

# ------------------------------------------------------------
# export high-resolution images
# ------------------------------------------------------------
pio.kaleido.scope.mathjax = None  # speed-up export

output_dir = os.path.join(Config.data_path, "images", "56bus")
os.makedirs(output_dir, exist_ok=True)

pdf_path = os.path.join(output_dir, "topology_loss.pdf")
png_path = os.path.join(output_dir, "topology_loss.png")

fig.write_image(pdf_path,  width=NATURE_CONFIG["width"],
                           height=NATURE_CONFIG["height"],
                           scale=2)   # 2 × 1800 px ≈ 300 dpi
fig.write_image(png_path,  width=NATURE_CONFIG["width"],
                           height=NATURE_CONFIG["height"],
                           scale=2)

average recovery step is:
14.625
15.063013476725034
average reactive power cost is:
54.58733743407025
101.6204096690605
the total cost is:
238.98502216379714
262.97381806591187


C:\Users\wdyao\AppData\Local\Temp\ipykernel_5000\3482050656.py:38: RuntimeWarning:

Mean of empty slice

C:\Users\wdyao\AppData\Local\Temp\ipykernel_5000\3482050656.py:39: RuntimeWarning:

All-NaN slice encountered

C:\Users\wdyao\AppData\Local\Temp\ipykernel_5000\3482050656.py:40: RuntimeWarning:

All-NaN slice encountered



In [34]:
### test our controller without topology error
voltage_ = []
q_ = []
cost_ = []
success_list_ = []
fail_list_ = []
entire_list_ = []
control_cost_ = []
reward_list_ = []
object_cost_list_ = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0) * 0.0
    # print(f'Topology for episode {episode}: {topology}')

    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    cost_ = []
    abnormal_stop = False

    for step in range(step_num):
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
            action_agent = 0.6* action_agent.detach().cpu().numpy()[0]
            action.append(action_agent)

        action = last_action - np.asarray(action)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            fail_list_.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            fail_list_.append((episode,step))
            abnormal_stop = True
            break
        if done:
            success_list_.append((episode,step))
            logger.success('episode {} stable at {} steps',success_list_[-1][0], success_list_[-1][1])
            break

        voltage_.append(state)

        q_.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost_.append(-reward)
        
        episode_control += LA.norm(action, 2)

    reward_list_.append(episode_reward)
    control_cost_.append(episode_control)
    object_cost_list_.append(np.sum(cost_))

    if (not done) and (abnormal_stop == False):
        entire_list_.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(success_list_))
logger.info('total fail episode is {}', len(fail_list_))
logger.info('number of finished at entire episode is {}', len(entire_list_))

success_list_ = np.array(success_list_)
print('average recovery step is:')
print(np.mean(success_list_[:,1]))
print(np.std(success_list_[:,1]))
print('average reactive power cost is:')
print(np.mean(control_cost_))
print(np.std(control_cost_))
print('the total cost is:')
print(np.mean(object_cost_list_))
print(np.std(object_cost_list_))

2025-06-12 11:35:37.658 | SUCCESS  | __main__:<module>:50 - episode 0 stable at 27 steps
2025-06-12 11:35:38.151 | SUCCESS  | __main__:<module>:50 - episode 1 stable at 10 steps
2025-06-12 11:35:39.520 | SUCCESS  | __main__:<module>:50 - episode 2 stable at 35 steps
2025-06-12 11:35:40.370 | SUCCESS  | __main__:<module>:50 - episode 3 stable at 20 steps
2025-06-12 11:35:41.431 | SUCCESS  | __main__:<module>:50 - episode 4 stable at 26 steps
2025-06-12 11:35:43.229 | SUCCESS  | __main__:<module>:50 - episode 5 stable at 44 steps
2025-06-12 11:35:43.605 | SUCCESS  | __main__:<module>:50 - episode 6 stable at 8 steps
2025-06-12 11:35:48.242 | SUCCESS  | __main__:<module>:50 - episode 7 stable at 116 steps
2025-06-12 11:35:51.044 | SUCCESS  | __main__:<module>:50 - episode 8 stable at 73 steps
2025-06-12 11:35:52.543 | SUCCESS  | __main__:<module>:50 - episode 9 stable at 38 steps
2025-06-12 11:35:54.447 | SUCCESS  | __main__:<module>:50 - episode 10 stable at 47 steps
2025-06-12 11:35:55.

average recovery step is:
31.585
28.96588985341206
average reactive power cost is:
120.46901415528887
232.34779806492892
the total cost is:
516.8304113980473
559.5291938315513


### baseline

In [10]:
### test the base line controller
num_agent = 5
voltage = []
q = []
base_cost = []
base_succ_list = []
base_fail_list = []
base_entire_list = []
base_control_cost = []
base_reward_list = []
base_object_cost = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    base_cost = []
    abnormal_stop = False

    for step in range(step_num):
        state1 = np.asarray(state-env.vmax + 0.001)
        state2 = np.asarray(env.vmin-state + 0.001)
        d_v = (np.maximum(state1, 0)-np.maximum(state2, 0)).reshape((num_agent,1))
        
        action = (last_action - 10*d_v)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            base_fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            base_fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            base_succ_list.append((episode,step))
            logger.success('stable at {}',base_succ_list[-1])
            break

        voltage.append(state)

        q.append(action)

        state = next_state
        
        episode_reward += reward
        
        base_cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    base_control_cost.append(episode_control)
    base_reward_list.append(episode_reward)
    base_object_cost.append(np.sum(base_cost))
    
    if (not done) and (abnormal_stop == False):
        base_entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(base_succ_list))
logger.info('total fail episode is {}', len(base_fail_list))
logger.info('number of finished at entire episode is {}', len(base_entire_list))

2025-03-19 17:01:31.851 | SUCCESS  | __main__:<module>:47 - stable at (0, 25)
2025-03-19 17:01:31.972 | SUCCESS  | __main__:<module>:47 - stable at (1, 7)
2025-03-19 17:01:32.412 | SUCCESS  | __main__:<module>:47 - stable at (2, 38)
2025-03-19 17:01:32.662 | SUCCESS  | __main__:<module>:47 - stable at (3, 22)
2025-03-19 17:01:32.902 | SUCCESS  | __main__:<module>:47 - stable at (4, 20)
2025-03-19 17:01:33.409 | SUCCESS  | __main__:<module>:47 - stable at (5, 46)
2025-03-19 17:01:33.466 | SUCCESS  | __main__:<module>:47 - stable at (6, 2)
2025-03-19 17:01:34.115 | SUCCESS  | __main__:<module>:47 - stable at (7, 59)
2025-03-19 17:01:34.829 | SUCCESS  | __main__:<module>:47 - stable at (8, 66)
2025-03-19 17:01:34.944 | SUCCESS  | __main__:<module>:47 - stable at (9, 10)
2025-03-19 17:01:35.428 | SUCCESS  | __main__:<module>:47 - stable at (10, 41)
2025-03-19 17:01:35.653 | SUCCESS  | __main__:<module>:47 - stable at (11, 20)
2025-03-19 17:01:35.811 | SUCCESS  | __main__:<module>:47 - stab

In [11]:
base_succ_list = np.array(base_succ_list)
print('average recovery step is:')
print(np.mean(base_succ_list[:,1]))
print(np.std(base_succ_list[:,1]))
print('average reactive power cost is:')
print(np.mean(base_control_cost))
print(np.std(base_control_cost))
print('the total cost is:')
print(base_object_cost)
print(np.mean(base_object_cost))
print(np.std(base_object_cost))


average recovery step is:
20.995
18.244861605394544
average reactive power cost is:
93.24602557422712
167.73421448165357
the total cost is:
[331.4381230096074, 79.37194620263368, 539.1592184469716, 284.34496406540705, 222.78735969737323, 754.1682702813467, 29.84013426138405, 1136.411871022985, 1056.9603629466496, 122.90933453666014, 560.9620343802666, 309.9329477329244, 162.46817297672234, 345.22781590281625, 84.00087648036495, 77.87157459061811, 447.9030283239126, 1143.0472313473442, 176.88333267800874, 557.5911198092685, 0.0, 91.96842769393515, 86.63425232033393, 337.90413282584814, 167.46888434182415, 81.27101095991333, 313.7137401372946, 213.78912549363267, 197.31651844449183, 233.30639714747372, 11.678091367223365, 208.69666857425483, 774.4340601730597, 310.12122161707396, 699.6066092539205, 165.95381847850743, 1365.3797876968981, 1149.4435999178938, 20.163409750434248, 889.4776413105868, 263.332872379289, 75.72796956047941, 242.8263436597923, 38.36772898430843, 253.7053771614978,

### Safe DDPG

In [7]:
### test the safe policy net
num_agent = 5
safe_voltage = []
safe_q = []
safe_cost = []
safe_succ_list = []
safe_fail_list = []
safe_entire_list = []
safe_contorl_cost = []
safe_reward_list = []
safe_object_cost = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    safe_cost = []
    abnormal_stop = False

    for step in range(step_num):
        action = []
        for i in range(num_agent):
            action_agent = safe_agent_net[i].get_action(torch.cuda.FloatTensor([state[i]]).float().reshape(1,1))
            action.append(action_agent)
        
        action = last_action - 5*np.asarray(action).reshape((num_agent, 1))
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            safe_fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            safe_fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            safe_succ_list.append((episode,step))
            logger.success('stable at {}',safe_succ_list[-1])
            break
        safe_voltage.append(state)

        safe_q.append(action)

        state = next_state
        
        episode_reward += reward
        
        safe_cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    safe_contorl_cost.append(episode_control)
    safe_reward_list.append(episode_reward)
    safe_object_cost.append(np.sum(safe_cost))

    if (not done) and (abnormal_stop == False):
        safe_entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(safe_succ_list))
logger.info('total fail episode is {}', len(safe_fail_list))
logger.info('number of finished at entire episode is {}', len(safe_entire_list))


2025-02-26 16:32:51.789 | SUCCESS  | __main__:<module>:48 - stable at (0, 32)
2025-02-26 16:32:51.944 | SUCCESS  | __main__:<module>:48 - stable at (1, 5)
2025-02-26 16:32:53.087 | SUCCESS  | __main__:<module>:48 - stable at (2, 53)
2025-02-26 16:32:53.631 | SUCCESS  | __main__:<module>:48 - stable at (3, 23)
2025-02-26 16:32:54.197 | SUCCESS  | __main__:<module>:48 - stable at (4, 21)
2025-02-26 16:32:55.642 | SUCCESS  | __main__:<module>:48 - stable at (5, 66)
2025-02-26 16:32:55.691 | SUCCESS  | __main__:<module>:48 - stable at (6, 0)
2025-02-26 16:32:57.145 | SUCCESS  | __main__:<module>:48 - stable at (7, 63)
2025-02-26 16:32:59.082 | SUCCESS  | __main__:<module>:48 - stable at (8, 83)
2025-02-26 16:32:59.340 | SUCCESS  | __main__:<module>:48 - stable at (9, 10)
2025-02-26 16:33:00.446 | SUCCESS  | __main__:<module>:48 - stable at (10, 51)
2025-02-26 16:33:00.898 | SUCCESS  | __main__:<module>:48 - stable at (11, 18)
2025-02-26 16:33:01.171 | SUCCESS  | __main__:<module>:48 - stab

In [8]:
safe_succ_list = np.array(safe_succ_list)
print('average recovery step is:')
print(np.mean(safe_succ_list[:,1]))
print(np.std(safe_succ_list[:,1]))
print('average reactive power cost is:')
print(np.mean(safe_contorl_cost))
print(np.std(safe_contorl_cost))
print('the total cost is:')
print(safe_object_cost)
print(np.mean(safe_object_cost))
print(np.std(safe_object_cost))

average recovery step is:
23.745
22.76510432657843
average reactive power cost is:
113.28480666367739
201.52561556901605
the total cost is:
[403.27795744816035, 55.325414340995955, 727.1358452981689, 294.4688507194921, 202.61152792137142, 1044.1712508603164, 0.0, 1231.4248697584305, 1315.2206187810293, 118.93681244404073, 670.783963796241, 274.1472899338466, 127.5456695089633, 423.3392394645283, 58.6587445928317, 0.0, 580.6666386539297, 1253.1081617403827, 184.43242772020008, 585.2495710662595, 0.0, 63.72166418513142, 64.9542762283673, 316.4960719270453, 150.98023253070193, 65.17465192296146, 332.1882578868361, 160.70081383701287, 194.89533951250965, 247.45159738431687, 0.0, 243.2038569518022, 1033.2732244977092, 363.1982795740444, 828.2557008552561, 132.2999422816827, 1487.2568119668974, 1252.4900974806337, 20.04220935187984, 1226.2190909731132, 260.4268165919477, 0.0, 186.11924126336132, 0.0, 353.5256759358745, 544.4987121291903, 937.807542688232, 28.16757670679894, 424.8058125176003